# Heads Down: Foreman Cost Forecaster, data and backtest

Mine-or-buy for a small nightly ORE rig, replayed over every ORE round since the Aug 12, 2026 fee change. Reproduce with `python3 fetch.py && .venv/bin/python backtest.py && .venv/bin/python forecast.py`. Written results: [RESULTS.md](RESULTS.md), model card: [MODEL_CARD.md](MODEL_CARD.md).

In [1]:
import json, warnings
import numpy as np, pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
SAMPLE = False
import orelib as L, backtest as BT, forecast as F, validate as V

## 1. Is the round reconstruction right?
Fee identities on every round, Motherlode pot vs every payout, replayed EMA vs api.ore.com and vs the live Board account.

In [2]:
val = V.run(sample=SAMPLE)
for k in ('coverage', 'fee_identities', 'motherlode', 'ema_vs_api', 'api_cost_below_price', 'chain_check', 'tiles'):
    print(k, json.dumps(val.get(k), indent=1, default=str)[:800])

coverage {
 "rounds": 58801,
 "first_round": 363816,
 "last_round": 422634,
 "first_time": "2026-08-11 23:38:15+00:00",
 "last_time": "2026-09-29 17:58:02+00:00",
 "missing_rounds": 18,
 "refund_rounds": 0,
 "mean_round_seconds": 71.65284013605442,
 "gaps": 8,
 "largest_gap_rounds": 11
}
fee_identities {
 "regime_start_used": "2026-08-13",
 "last_round_on_old_rules": "2026-08-12 22:55:47+00:00",
 "admin_1pct_share_in_regime": 1.0,
 "protocol_10pct_losing_share_in_regime": 1.0,
 "minted_1p2_share_in_regime": 1.0,
 "rounds_in_regime": 57718
}
motherlode {
 "hits": 127,
 "max_abs_error_ore": 0.0,
 "mean_payout_ore": 94.54330708661418,
 "hit_rate": 0.0021598272138228943,
 "expected_hit_rate": 0.002
}
ema_vs_api {
 "snapshots": 167,
 "median_ratio": 0.9998077057302033,
 "p05": 0.9936510767869181,
 "p95": 1.0046645549403166,
 "note": "api.ore.com production_cost (USD) / SOL-USD from pool candles vs replayed EMA"
}
api_cost_below_price {
 "below": 149,
 "of": 167
}
chain_check [
 {
  "round_i

## 2. Crowding follows the Motherlode pot
`production_cost_ema` counts 1.2 ORE per round and ignores the pot, but miners pile in when the pot is large. Adjusting the EMA for the pot's expected value (1 + pot/500 ORE per round) flattens the cost curve.

In [3]:
pd.DataFrame(val['crowding_vs_pot']['by_pot']).T.round(3)

,rounds,sol_deployed,pot,ema_sol,ev_cost_sol
"(0, 25]",13220.0,6.364,12.065,0.516,0.604
"(25, 50]",10318.0,6.529,36.965,0.514,0.574
"(50, 100]",12680.0,6.820,72.256,0.537,0.563
"(100, 150]",7457.0,7.291,123.402,0.574,0.553
"(150, 200]",4876.0,7.738,172.807,0.608,0.542
"(200, 300]",5874.0,8.881,244.907,0.698,0.563
"(300, 10000]",3293.0,11.285,376.971,0.889,0.608


## 3. Mine-or-buy at small budgets
![](figures/eff_price_vs_budget.png)

In [4]:
window = L.ShiftWindow(23, 7, 1)
prep = BT.prepare(SAMPLE, window)
r, n_med = prep['rounds'], prep['rounds_per_night']
costs = L.Costs()
print(f"{r['night'].nunique()} nights, median {n_med} rounds/night")
tables = {}
for B in (0.02, 0.04, 0.05):
    A = B * 1e9 / n_med
    tables[B] = BT.summarize(BT.evaluate(r, A, costs, BT.decisions_for(r, A, costs)), r, costs, n_boot=500)
tables[0.04][['eff_sol_per_ore', 'vs_always_buy_pct', 'ci95_lo_pct', 'ci95_hi_pct', 'mined_round_share']].round(4)

47 nights, median 378 rounds/night


,eff_sol_per_ore,vs_always_buy_pct,ci95_lo_pct,ci95_hi_pct,mined_round_share
strategy,,,,,
always_buy,0.6752,0.0000,0.0000,0.0000,0.0000
always_mine,0.9817,45.3971,42.9737,47.7802,1.0000
gate_naive,0.9250,37.0057,32.1520,41.5444,0.8886
gate_ema,0.6752,0.0000,0.0000,0.0000,0.0000
gate_ev,0.6752,0.0000,0.0000,0.0000,0.0000
gate_ema_hourly,0.6752,0.0000,0.0000,0.0000,0.0000
gate_ev_hourly,0.6752,0.0000,0.0000,0.0000,0.0000
gate_ev_arm,0.6752,0.0000,0.0000,0.0000,0.0000
oracle_hour,0.6752,0.0000,0.0000,0.0000,0.0000


## 4. Variance of a small rig
![](figures/ore_per_night.png)

In [5]:
rows = []
for B in (0.02, 0.04):
    A = B * 1e9 / n_med
    for pol in L.TILE_POLICIES:
        d = BT.night_distribution(r, A, costs, pol, n_sims=1000)
        d.pop('samples'); d['budget'] = B; rows.append(d)
pd.DataFrame(rows)[['budget', 'policy', 'mean_ore', 'p5', 'median', 'p95', 'p_zero']]

,budget,policy,mean_ore,p5,median,p95,p_zero
0,0.02,all25,0.003275,0.001108,0.001912,0.003552,0.000000
1,0.02,split15,0.003218,0.001846,0.003108,0.005397,0.000000
2,0.02,solo10,0.003361,0.000000,0.000000,0.003400,0.678894
3,0.02,rand5,0.003275,0.001051,0.001711,0.006350,0.000000
4,0.04,all25,0.006550,0.002216,0.003823,0.007105,0.000000
5,0.04,split15,0.006435,0.003693,0.006215,0.010794,0.000000
6,0.04,solo10,0.006722,0.000000,0.000000,0.006800,0.676638
7,0.04,rand5,0.006550,0.002102,0.003423,0.012809,0.000000


## 5. Forecaster: does predicting the EMA beat reading it?
![](figures/forecast_walkforward.png)

In [6]:
rounds = L.load_rounds(sample=SAMPLE); prices = L.load_prices(sample=SAMPLE)
since = str(rounds['time'].min().date()) if SAMPLE else BT.REGIME_START
h = F.add_targets(F.hourly_table(rounds, prices, since), (1,))
pred = F.walk_forward(h, 1, since, min_train_days=1 if SAMPLE else 7)
pd.DataFrame(F.score(pred)).T.round(4) if len(pred) else 'not enough data'

,mae_log,rmse_log,mae_skill_vs_persistence,sign_accuracy,n
persistence,0.0259,0.0415,0.0000,0.9713,976.0
ridge,0.0235,0.0360,0.0922,0.9795,976.0
gbm,0.0242,0.0379,0.0663,0.9754,976.0


See [RESULTS.md](RESULTS.md) for the recommendation.